# Which arm won?

`attest runs compare` ranks the arms of a sweep and then says why you may not
trust the ranking. **The caveats are the product, not a disclaimer**: a winner
reported without them misrepresents what was found. This notebook compares the
`kdsweep` distillation sweep and shows its caveats exactly as the tool printed
them.

It runs offline (no model server) against a temporary ledger.

In [1]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# Everything this notebook writes goes to a throwaway folder and a throwaway
# ledger. Your real database and workspace are never read or changed.
WORK = Path(tempfile.mkdtemp(prefix="attest-demo-"))
os.environ["ATTEST_DB"] = str(WORK / "demo.db")
os.environ["HERMES_HOME"] = str(WORK / "hermes")  # so your own metric_direction.toml is not read
for _var in ("RESEARCH_ROOT", "LEDGER_METRIC_DIRECTION_FILE", "LEDGER_CORPUS_FILE"):
    os.environ.pop(_var, None)  # so nothing of yours decides how this demo ranks


def _find_attest():
    found = shutil.which("attest", path=str(Path(sys.executable).parent)) or shutil.which("attest")
    if found is None:
        raise RuntimeError("`attest` is not installed for this Python: pip install attestation")
    return found


_ATTEST = _find_attest()


def attest(command, cwd):
    """Run `attest <command>` offline in `cwd`, print what it prints, return its exit code."""
    print("$ attest " + command)
    done = subprocess.run(
        [_ATTEST, *shlex.split(command)], cwd=cwd, capture_output=True, text=True, timeout=60
    )
    text = (done.stdout + done.stderr).rstrip()
    print(text.replace(str(WORK), "<tmp>").replace(str(Path.home()), "~"))
    print(f"[exit {done.returncode}]")
    return done.returncode


def _run(tag, wer, val_loss):
    record = {"tag": tag, "corpus": "librispeech-100h", "wer": wer, "val_loss": val_loss}
    return json.dumps({**record, "n_records": 2620})


FALLBACK = {
    "speech-distill/results/kdsweep_baseline.json": _run("kdsweep_baseline", 0.0731, 2.4),
    "speech-distill/results/kdsweep_t2.json": _run("kdsweep_t2", 0.0688, 2.3),
    "speech-distill/results/kdsweep_t4.json": _run("kdsweep_t4", 0.0642, 2.18),
    "speech-distill/results/kdsweep_t4b.json": _run("kdsweep_t4b", 0.0659, 2.2),
    "speech-distill/FINDINGS.md": """# Distillation notes

The baseline reaches a word error rate of 0.0731.
<!-- claim: speech-distill/kdsweep_baseline metric=wer value=0.0731 -->

Temperature 4 is better, at 0.0642.
<!-- claim: speech-distill/kdsweep_t4 metric=wer value=0.0642 -->

Temperature 2 gives 0.0701 (stale: the run now says otherwise).
<!-- claim: speech-distill/kdsweep_t2 metric=wer value=0.0701 -->

Temperature 8 gives 0.0600 (there is no such run).
<!-- claim: speech-distill/kdsweep_t8 metric=wer value=0.0600 -->

A claim with no metric at all.
<!-- claim: speech-distill/kdsweep_baseline value=0.0731 -->

Validation loss ended at 2.18 and the model has 41.3M parameters.
""",
}


def find_example_workspace():
    """examples/workspace from a repo checkout, or None. ATTEST_EXAMPLE_WORKSPACE
    points at one explicitly; the value `none` skips the search."""
    override = os.environ.get("ATTEST_EXAMPLE_WORKSPACE")
    if override == "none":
        return None
    if override:
        return Path(override)
    import attestation

    for base in (Path.cwd(), Path(attestation.__file__).resolve().parent):
        for folder in (base, *base.parents):
            candidate = folder / "examples" / "workspace"
            if (candidate / "speech-distill" / "FINDINGS.md").is_file():
                return candidate
    return None


WORKSPACE = WORK / "workspace"
SOURCE = find_example_workspace()
if SOURCE is not None:
    shutil.copytree(SOURCE, WORKSPACE)
    print("Using the example workspace from the attestation repo (copied to a temporary folder).")
else:
    for name, text in FALLBACK.items():
        target = WORKSPACE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)
    print("The repo's examples/workspace is not on this machine, so this notebook")
    print("wrote a small stand-in with the same shape (also in a temporary folder).")

Using the example workspace from the attestation repo (copied to a temporary folder).


## 1. Read the runs into the ledger

A *family* is a shared filename prefix (`kdsweep_t2`, `kdsweep_t4` ... form the
family `kdsweep`), not a project. `runs list` shows the families that exist.

In [2]:
attest("runs scan --root .", cwd=WORKSPACE)
attest("runs list", cwd=WORKSPACE)

$ attest runs scan --root .


  retrieval-ablation           5 run(s)
  speech-distill               4 run(s)
9 run(s) across 2 project(s)
[exit 0]
$ attest runs list


  retrieval-ablation   planned_colbert                        spec       [planned]
  retrieval-ablation   rank_method_bm25                       recorded   [rank-method]
  retrieval-ablation   rank_method_dense                      recorded   [rank-method]
  retrieval-ablation   rank_method_dense2                     recorded   [rank-method]
  retrieval-ablation   rank_method_hybrid                     recorded   [rank-method]
  speech-distill       kdsweep_baseline                       recorded   [kdsweep]
  speech-distill       kdsweep_t2                             recorded   [kdsweep]
  speech-distill       kdsweep_t4                             recorded   [kdsweep]
  speech-distill       kdsweep_t4b                            recorded   [kdsweep]

  family kdsweep                          4 run(s)  (speech-distill)
  family rank-method                      4 run(s)  (retrieval-ablation)
  family planned                          1 run(s)  (retrieval-ablation)
[exit 0]


0

## 2. Compare by a named metric

Word error rate is better when **lower**. The ledger knows that direction, so
it ranks. Pass the metric explicitly: with none, it falls back to whichever
metric most arms share, which can answer a different question than you asked.

In [3]:
attest("runs compare kdsweep --metric wer", cwd=WORKSPACE)

$ attest runs compare kdsweep --metric wer


kdsweep — ranked by wer (lower_is_better), all arms on librispeech-100h

  arm                                                 wer      n      step  source
  -------------------------------------------- ---------- ------  --------  ------
  kdsweep_t4                                       0.0642   2620            speech-distill/results/kdsweep_t4.json
  kdsweep_t4b                                      0.0659   2620            speech-distill/results/kdsweep_t4b.json
  kdsweep_t2                                       0.0688   2620            speech-distill/results/kdsweep_t2.json
  kdsweep_baseline                                 0.0731   2620            speech-distill/results/kdsweep_baseline.json

winner: kdsweep_t4
  caveat: the top two arms differ by 0.0017 (2.6%) -- too close to call from these numbers alone
  caveat: each arm is a single run; no seed replication, so this ranking cannot separate configuration from run-to-run variance
[exit 0]


0

## 3. Read the caveats, not just the winner

The two lines starting `caveat:` above are verbatim from the tool. In words:

- **Too close to call.** The top two arms differ by 0.0017 (2.6%). In this
  sweep `kdsweep_t4b` is `kdsweep_t4` at a different seed, so 0.0017 is
  also roughly the size of run-to-run noise.
- **No seed replication.** Each arm is one run, so the ranking cannot separate
  the configuration from luck.

A comparison whose margin is smaller than its seed variance has not found
anything. The honest summary is "t4 is nominally ahead; this sweep cannot tell
t4 from t2 or t4b", not "t4 won".

## 4. When it refuses

The ledger **refuses to rank a metric whose direction nobody declared**, rather
than guess: ranking WER as if higher were better would name the worst arm the
winner. The repo's example has a second sweep to show it (this cell only runs
when the repo's examples are present).

In [4]:
if SOURCE is not None:
    attest("runs scan --root .", cwd=WORKSPACE)
    attest("runs compare rank-method --metric n_records", cwd=WORKSPACE)
else:
    print("Skipped: needs the repo's retrieval-ablation example.")

$ attest runs scan --root .


  retrieval-ablation           5 run(s)
  speech-distill               4 run(s)
9 run(s) across 2 project(s)
[exit 0]
$ attest runs compare rank-method --metric n_records


unknown direction for metric 'n_records' -- refusing to rank. Declare it under [metric_direction] in <tmp>/hermes/metric_direction.toml; guessing would rank ablation arms backwards.
[exit 1]


The refusal is the right answer (here the file it names sits in a temporary
home; on your machine it is `~/.hermes/metric_direction.toml`). The fix is a
`[metric_direction]` entry made by the person who knows which way is better, never a guess by the tool or
by an agent reporting it.